# Pipeline Pembersihan & Standarisasi Data Penjualan (Data Cleaning Pipeline)
**Kasus Bisnis:** Warung Bakso Lintang (Ketintang, Surabaya)  
**Dokumen Acuan:** [PRD.md](PRD.md) v1.1 - Sistem Prediksi Stok & Business Intelligence  
**Penyempurnaan Dataset Bersih:**
1. Kategori **"0-Nagih"** diabaikan dan dibuang ke `dropped_nagih.csv`.
2. Kolom redundan **`Date` dan `Time` dihapus** (karena informasi tanggal dan jam sudah $100\%$ terwakili secara presisi oleh kolom tunggal **`Datetime`**).
3. Kolom **`Item_Flag`** ditambahkan untuk menandai menu khusus (`Bundle` dan `Custom Amount`).
4. Kolom **`Customer` (nomor HP) dihapus** dari semua berkas output demi privasi pelanggan dan efisiensi model/BI.

---

### Prinsip Utama Data Cleaning:
1. **Tidak Menimpa File Asli:** Seluruh hasil transformasi disimpan ke file terpisah (`sales_clean.csv`, `refunds.csv`, `dropped_nagih.csv`, dan file siap-pakai).
2. **Audit Lengkap (No Silent Drops):** Baris yang dikeluarkan diarsipkan secara transparan dengan alasan terdokumentasi.
3. **Integritas Finansial (Zero Loss):** Nilai uang (Gross Sales, Discounts, Refunds, Net Sales) diawasi ketat lewat pengujian `assert`.
4. **Struktur Modular:** Setiap langkah terpisah secara granular disertai pencatatan dimensi dataset (sebelum vs sesudah).


## 1. Membaca CSV, Standarisasi Tanggal/Waktu, & Pembuangan Transaksi Nagih


### 1.1 Membaca File Input CSV & Menyimpan Salinan Mentah
Membaca dataset POS penjualan warung bakso dari file `sales_report.csv` dan mencatat dimensi data awal.


In [1]:
# 1.1 Membaca file input dan inisialisasi DataFrame
import pandas as pd
import numpy as np

input_file = 'sales_report.csv'
df_raw = pd.read_csv(input_file)
df = df_raw.copy()

# Hitung rentang tanggal transaksi dari datetime (format DD-MM-YYYY)
_dates = pd.to_datetime(df['Date'], format='%d-%m-%Y')
date_min_str = _dates.min().strftime('%d-%m-%Y')
date_max_str = _dates.max().strftime('%d-%m-%Y')

print(f"[1.1] Dataset awal berhasil dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print(f"      Rentang tanggal transaksi: {date_min_str} s/d {date_max_str}")


[1.1] Dataset awal berhasil dimuat: 64,327 baris x 25 kolom
      Rentang tanggal transaksi: 01-01-2026 s/d 24-09-2026


### 1.2 Membentuk Kolom `Datetime` Gabungan dari `Date` dan `Time`
Menggabungkan informasi tanggal dan waktu ke dalam satu kolom penanda waktu presisi bertipe `datetime64[ns]`.


In [2]:
# 1.2 Membuat kolom Datetime lengkap dari Date + Time
print(f"[1.2] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

# Standarisasi format tanggal dari DD-MM-YYYY ke format Datetime ISO
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d-%m-%Y %H:%M:%S',
    errors='coerce'
)

print(f"[1.2] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah kolom 'Datetime')")
print(f"      Contoh Datetime pertama: {df['Datetime'].iloc[0]}")
print(f"      Tipe data Datetime: {df['Datetime'].dtype}")


[1.2] SEBELUM : 64,327 baris | 25 kolom
[1.2] SESUDAH : 64,327 baris | 26 kolom (tambah kolom 'Datetime')
      Contoh Datetime pertama: 2026-09-24 21:52:08
      Tipe data Datetime: datetime64[ns]


### 1.3 Pembuangan & Pengarsipan Seluruh Struk Kategori "0-Nagih" (Keputusan Owner)
Kategori 0-Nagih adalah uji coba penjualan di warkop (21 Feb-13 Mar 2026) yang tidak dilanjutkan, berdasarkan konfirmasi Owner. Karena bukan penjualan operasional warung, seluruh struk terkait dikeluarkan dan diarsipkan di `dropped_nagih.csv`. Awalan struk `9XT` (kasir `Reny Komar`) dan `8AA` (kasir `Insan Admin`) adalah bagian dari uji coba ini.

Langkah ini membuang **seluruh baris** dari setiap struk (`Receipt Number`) yang mengandung minimal 1 item `0-Nagih` (termasuk item non-Nagih yang berada di struk yang sama dan baris refund Nagih), lalu mengarsipkannya ke `dropped_nagih.csv`.

In [3]:
# 1.3 Mengidentifikasi dan membuang seluruh struk yang memuat kategori 0-Nagih
print(f"[1.3] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

# Identifikasi semua Receipt Number yang memiliki minimal satu baris Category == '0-Nagih'
nagih_receipts = df.loc[df['Category'] == '0-Nagih', 'Receipt Number'].unique()
num_nagih_receipts = len(nagih_receipts)

# Pisahkan baris yang dibuang sebagai arsip (hapus kolom Customer pada file simpanan demi privasi)
dropped_nagih = df[df['Receipt Number'].isin(nagih_receipts)].copy()
file_dropped_nagih = 'dropped_nagih.csv'
dropped_nagih.drop(columns=['Customer']).to_csv(file_dropped_nagih, index=False)

# Hitung rincian baris yang dibuang
count_nagih_cat = (dropped_nagih['Category'] == '0-Nagih').sum()
count_non_nagih_cat = (dropped_nagih['Category'] != '0-Nagih').sum()
dropped_gross = dropped_nagih['Gross Sales'].sum()
dropped_net = dropped_nagih['Net Sales'].sum()

# Update DataFrame utama dengan mengecualikan struk Nagih
df = df[~df['Receipt Number'].isin(nagih_receipts)].copy()

print(f"[1.3] PEMBUANGAN STRUK NAGIH SELESAI:")
print(f"      - Jumlah struk Nagih teridentifikasi : {num_nagih_receipts:,} struk")
print(f"      - Total baris dibuang                : {len(dropped_nagih):,} baris")
print(f"        * Item kategori '0-Nagih'          : {count_nagih_cat:,} baris")
print(f"        * Item non-Nagih di struk Nagih    : {count_non_nagih_cat:,} baris")
print(f"      - Total Gross Sales dibuang          : Rp {dropped_gross:,.2f}")
print(f"      - Total Net Sales dibuang            : Rp {dropped_net:,.2f}")
print(f"      - Berkas arsip tersimpan (tanpa Customer): '{file_dropped_nagih}'")
print(f"[1.3] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom")

# Validasi assert ketat
assert (df['Category'] == '0-Nagih').sum() == 0, "Masih ditemukan kategori '0-Nagih' di dataset!"
assert df['Receipt Number'].isin(nagih_receipts).sum() == 0, "Masih ditemukan Receipt Number Nagih di dataset!"
print("      [ASSERT PASS] Kategori 0-Nagih dan seluruh struk terkait telah bersih 100%.")


[1.3] SEBELUM : 64,327 baris | 26 kolom
[1.3] PEMBUANGAN STRUK NAGIH SELESAI:
      - Jumlah struk Nagih teridentifikasi : 98 struk
      - Total baris dibuang                : 392 baris
        * Item kategori '0-Nagih'          : 382 baris
        * Item non-Nagih di struk Nagih    : 10 baris
      - Total Gross Sales dibuang          : Rp 1,621,000.00
      - Total Net Sales dibuang            : Rp 1,393,000.07
      - Berkas arsip tersimpan (tanpa Customer): 'dropped_nagih.csv'
[1.3] SESUDAH : 63,935 baris | 26 kolom
      [ASSERT PASS] Kategori 0-Nagih dan seluruh struk terkait telah bersih 100%.


## 2. Drop Kolom Tidak Berguna / Redundan & Rekayasa Fitur Waktu


### 2.1 Menghapus Kolom Tidak Berguna, Redundan (`Date`, `Time`), Kosong, & Data Sensitif (`Customer`)
Menghapus kolom-kolom yang tidak memberikan nilai informasi atau bersifat sensitif:
- **`Customer` (Data Sensitif)**: Dihapus total dari seluruh output demi menjaga privasi data pelanggan, dan tidak dibutuhkan untuk peramalan bahan baku maupun business intelligence.
- **`Date` & `Time` (Redundan)**: Dihapus karena tanggal dan jam sudah tersimpan lengkap dan presisi di kolom **`Datetime`**.
- `Outlet`: Bernilai konstan 100% ('Lintang Bakso Ketintang').
- `Gratuity` & `Tax`: Seluruh baris bernilai 0.
- `Collected By`: >99% satu orang kasir ('Risa Yan').
- `Brand`: Redundan dengan kolom `Category`.
- `Variant`, `SKU`, `Served By`: 100% bernilai kosong (NaN) dari sistem kasir.

In [4]:
# 2.1 Drop kolom yang tidak berguna, redundan, kosong, dan data sensitif
print(f"[2.1] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

drop_cols = ['Outlet', 'Gratuity', 'Tax', 'Collected By', 'Brand', 'Date', 'Time', 'Customer'] + [c for c in ['Variant', 'SKU', 'Served By'] if c in df.columns]
df = df.drop(columns=drop_cols)

print(f"[2.1] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (berhasil drop {len(drop_cols)} kolom)")
print(f"      Kolom yang di-drop: {drop_cols}")


[2.1] SEBELUM : 63,935 baris | 26 kolom
[2.1] SESUDAH : 63,935 baris | 15 kolom (berhasil drop 11 kolom)
      Kolom yang di-drop: ['Outlet', 'Gratuity', 'Tax', 'Collected By', 'Brand', 'Date', 'Time', 'Customer', 'Variant', 'SKU', 'Served By']


### 2.2 Rekayasa Fitur Waktu dari Kolom `Datetime`
Membuat fitur waktu analitis langsung dari kolom **`Datetime`** untuk analisis berkala dan variabel model prediksi:
- `Year_Month`: Periode tahun-bulan (misal: 2026-01).
- `Day_Name`: Nama hari (Senin s/d Minggu).
- `Hour`: Jam transaksi ($0$ s/d $23$).
- `Is_Weekend`: Indikator akhir pekan (Sabtu & Minggu).

In [5]:
# 2.2 Mengekstrak atribut waktu berbasis Datetime
print(f"[2.2] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

df['Year_Month'] = df['Datetime'].dt.to_period('M')
df['Day_Name'] = df['Datetime'].dt.day_name()
df['Hour'] = df['Datetime'].dt.hour
df['Is_Weekend'] = df['Datetime'].dt.dayofweek.isin([5, 6])

print(f"[2.2] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah 4 kolom fitur waktu)")
print(f"      Contoh fitur waktu: Year_Month={df['Year_Month'].iloc[0]}, Day_Name={df['Day_Name'].iloc[0]}, Hour={df['Hour'].iloc[0]}, Is_Weekend={df['Is_Weekend'].iloc[0]}")


[2.2] SEBELUM : 63,935 baris | 15 kolom
[2.2] SESUDAH : 63,935 baris | 19 kolom (tambah 4 kolom fitur waktu)
      Contoh fitur waktu: Year_Month=2026-09, Day_Name=Thursday, Hour=21, Is_Weekend=False


## 3. Penanganan Baris Refund (Event Type == "Refund")


### 3.1 Penandaan Baris Refund (`Is_Refund`)
Menambahkan kolom boolean `Is_Refund` untuk memudahkan filter transaksi pembayaran vs pengembalian dana.


In [6]:
# 3.1 Menambahkan kolom flag Is_Refund
print(f"[3.1] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

df['Is_Refund'] = df['Event Type'] == 'Refund'
num_refunds = df['Is_Refund'].sum()

print(f"[3.1] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah 'Is_Refund')")
print(f"      Jumlah baris refund terdeteksi: {num_refunds:,} baris")


[3.1] SEBELUM : 63,935 baris | 19 kolom
[3.1] SESUDAH : 63,935 baris | 20 kolom (tambah 'Is_Refund')
      Jumlah baris refund terdeteksi: 149 baris


### 3.2 Ekstraksi `Refund_Type` dan Pembersihan Nama Item Asli
Kolom `Items` pada baris refund berformat `Full Refund for: <struk> - <nama item>` atau `Partial Refund for: ...`.
Informasi tipe refund diekstrak ke kolom `Refund_Type` ('Full'/'Partial'/'Tidak Ada'), dan nama menu aslinya dipulihkan ke kolom `Items`.

In [7]:
# 3.2 Ekstraksi Refund_Type dan pembersihan kolom Items
print(f"[3.2] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

def extract_refund_info(row):
    if not row['Is_Refund']:
        return 'Tidak Ada', row['Items']
    item_str = str(row['Items'])
    if 'Full Refund for:' in item_str:
        clean_item = item_str.split(' - ', 1)[1] if ' - ' in item_str else item_str.replace('Full Refund for:', '').strip()
        return 'Full', clean_item.strip()
    elif 'Partial Refund for:' in item_str:
        clean_item = item_str.split(' - ', 1)[1] if ' - ' in item_str else item_str.replace('Partial Refund for:', '').strip()
        return 'Partial', clean_item.strip()
    else:
        return 'Other', item_str.strip()

refund_info = df.apply(extract_refund_info, axis=1)
df['Refund_Type'] = [x[0] for x in refund_info]
df['Items'] = [x[1] for x in refund_info]

print(f"[3.2] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah 'Refund_Type' & bersihkan 'Items')")
print(f"      Distribusi Refund_Type:")
print(df['Refund_Type'].value_counts())


[3.2] SEBELUM : 63,935 baris | 20 kolom
[3.2] SESUDAH : 63,935 baris | 21 kolom (tambah 'Refund_Type' & bersihkan 'Items')
      Distribusi Refund_Type:
Refund_Type
Tidak Ada    63786
Full           114
Partial         35
Name: count, dtype: int64


### 3.3 Pemisahan Baris Refund ke File Tersendiri (`refunds.csv`)
Mengekspor baris refund ke file audit terpisah `refunds.csv` tanpa menghapus baris refund dari dataset utama.


In [8]:
# 3.3 Menyimpan baris refund ke file terpisah refunds.csv
file_refunds = 'refunds.csv'
df_refunds = df[df['Is_Refund']].copy()
df_refunds.to_csv(file_refunds, index=False)

print(f"[3.3] File audit refund tersimpan di '{file_refunds}': {df_refunds.shape[0]:,} baris x {df_refunds.shape[1]} kolom")
print(f"      Total Net Sales Refund: Rp {df_refunds['Net Sales'].sum():,.2f}")


[3.3] File audit refund tersimpan di 'refunds.csv': 149 baris x 21 kolom
      Total Net Sales Refund: Rp -1,519,459.26


## 4. Penanganan Baris dengan Atribut Identik Menggunakan Agregasi (Bukan Drop Mentah)

### 4.1 Identifikasi Baris dengan Atribut Identik (Kuantitas/Nilai Boleh Berbeda)
Mengecek baris dengan atribut identik dalam satu struk (diduga pemesanan bertahap, belum dikonfirmasi Owner).

In [9]:
# 4.1 Menghitung baris duplikat persis dan baris berlebih yang akan digabung
num_exact_all_cols = df.duplicated().sum()

metric_cols = ['Quantity', 'Gross Sales', 'Discounts', 'Refunds', 'Net Sales']
group_cols = [c for c in df.columns if c not in metric_cols]

num_excess_group = df.duplicated(subset=group_cols, keep='first').sum()
num_total_involved = df.duplicated(subset=group_cols, keep=False).sum()

print(f"[4.1] Pemeriksaan Baris Duplikat:")
print(f"      (a) Baris duplikat persis di SEMUA kolom (termasuk metrik) : {num_exact_all_cols:,} baris")
print(f"      (b) Baris berlebih dengan atribut identik yang akan digabung : {num_excess_group:,} baris")
print(f"          (Total baris yang terlibat dalam penggabungan        : {num_total_involved:,} baris)")


[4.1] Pemeriksaan Baris Duplikat:
      (a) Baris duplikat persis di SEMUA kolom (termasuk metrik) : 1,837 baris
      (b) Baris berlebih dengan atribut identik yang akan digabung : 2,504 baris
          (Total baris yang terlibat dalam penggabungan        : 4,520 baris)


### 4.2 Agregasi Baris Duplikat (Menjumlahkan Metrik Finansial & Kuantitas)
Menggabungkan baris duplikat dengan menjumlahkan `Quantity`, `Gross Sales`, `Discounts`, `Refunds`, dan `Net Sales`, serta mencatat `Merged_Row_Count`.


In [10]:
# 4.2 Menjalankan agregasi duplikat
print(f"[4.2] SEBELUM AGREGASI : {df.shape[0]:,} baris | {df.shape[1]} kolom")
gross_before_agg = df['Gross Sales'].sum()
net_before_agg = df['Net Sales'].sum()
qty_before_agg = df['Quantity'].sum()
rows_before_agg = len(df)

# Inisialisasi hitungan baris asli sebelum digabung
df['Merged_Row_Count'] = 1
agg_metric_cols = metric_cols + ['Merged_Row_Count']

agg_rules = {col: 'sum' for col in agg_metric_cols}
df_aggregated = df.groupby(group_cols, dropna=False, as_index=False).agg(agg_rules)

# Urutkan kolom agar rapi
cols_sorted = group_cols + agg_metric_cols
df = df_aggregated[cols_sorted].copy()

print(f"[4.2] SESUDAH AGREGASI : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah 'Merged_Row_Count')")
print(f"      Baris yang berhasil dirampingkan: {df_raw.shape[0] - df.shape[0]:,} baris dari raw")


[4.2] SEBELUM AGREGASI : 63,935 baris | 21 kolom
[4.2] SESUDAH AGREGASI : 61,431 baris | 22 kolom (tambah 'Merged_Row_Count')
      Baris yang berhasil dirampingkan: 2,896 baris dari raw


### 4.3 Verifikasi Integritas Finansial Pasca-Agregasi (`assert`)
Memastikan total nominal Gross Sales dan Net Sales sebelum dan sesudah agregasi bernilai sama persis tanpa selisih.


In [11]:
# 4.3 Validasi ketat bahwa total uang tidak berkurang atau bertambah
gross_after_agg = df['Gross Sales'].sum()
net_after_agg = df['Net Sales'].sum()
qty_after_agg = df['Quantity'].sum()

diff_gross = abs(gross_before_agg - gross_after_agg)
diff_net = abs(net_before_agg - net_after_agg)
diff_qty = abs(qty_before_agg - qty_after_agg)

print(f"[4.3] Gross Sales Sebelum: Rp {gross_before_agg:,.2f} | Sesudah: Rp {gross_after_agg:,.2f} | Selisih: Rp {diff_gross:.4f}")
print(f"      Net Sales Sebelum  : Rp {net_before_agg:,.2f} | Sesudah: Rp {net_after_agg:,.2f} | Selisih: Rp {diff_net:.4f}")
print(f"      Quantity Sebelum   : {qty_before_agg:,} | Sesudah: {qty_after_agg:,} | Selisih: {diff_qty}")
print(f"      Merged_Row_Count Sum: {df['Merged_Row_Count'].sum():,} | Baris Sebelum: {rows_before_agg:,}")

assert diff_gross < 0.01, f"Total Gross Sales berubah pasca-agregasi! Selisih = {diff_gross}"
assert diff_net < 0.01, f"Total Net Sales berubah pasca-agregasi! Selisih = {diff_net}"
assert qty_before_agg == qty_after_agg, f"Total Quantity berubah pasca-agregasi! Selisih = {diff_qty}"
assert df['Merged_Row_Count'].sum() == rows_before_agg, "Total Merged_Row_Count tidak cocok dengan baris sebelum agregasi!"
print("      [ASSERT PASS] Integritas finansial, kuantitas, dan rekam baris 100% terjaga sempurna.")


[4.3] Gross Sales Sebelum: Rp 577,677,500.00 | Sesudah: Rp 577,677,500.00 | Selisih: Rp 0.0000
      Net Sales Sebelum  : Rp 544,329,042.52 | Sesudah: Rp 544,329,042.52 | Selisih: Rp 0.0000
      Quantity Sebelum   : 98,181 | Sesudah: 98,181 | Selisih: 0
      Merged_Row_Count Sum: 63,935 | Baris Sebelum: 63,935
      [ASSERT PASS] Integritas finansial, kuantitas, dan rekam baris 100% terjaga sempurna.


## 5. Penandaan Anomali Transaksi (`Anomaly_Flag`)


### 5.1 Deteksi dan Penandaan Anomali
Menandai transaksi khusus ke kolom `Anomaly_Flag` (tanpa menghapus baris):
- `Gross Sales >= 500.000`: **"Outlier Gross"** (transaksi porsi luar biasa besar).
- Reason of Refund memuat kata trial/belajar: **"Kemungkinan Uji Coba"**.
- `Net Sales == 0` dan Event Type == 'Payment': **"Gratis/Diskon Penuh"**.
- Transaksi normal: **"Normal"**.

In [12]:
# 5.1 Menetapkan nilai Anomaly_Flag
print(f"[5.1] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

def detect_anomaly(row):
    if row['Gross Sales'] >= 500000:
        return 'Outlier Gross'
    reason = str(row.get('Reason of Refund', '')).lower()
    if 'trial' in reason or 'belajar' in reason:
        return 'Kemungkinan Uji Coba'
    if row['Net Sales'] == 0 and row['Event Type'] == 'Payment':
        return 'Gratis/Diskon Penuh'
    return 'Normal'

df['Anomaly_Flag'] = df.apply(detect_anomaly, axis=1)

print(f"[5.1] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah kolom 'Anomaly_Flag')")


[5.1] SEBELUM : 61,431 baris | 22 kolom
[5.1] SESUDAH : 61,431 baris | 23 kolom (tambah kolom 'Anomaly_Flag')


### 5.2 Pemeriksaan Distribusi `Anomaly_Flag` & Transaksi Outlier
Mengecek distribusi anomali dan memeriksa detail baris `Outlier Gross`.


In [13]:
# 5.2 Cek sebaran Anomaly_Flag dan inspeksi Outlier Gross
print(f"[5.2] Distribusi Kolom Anomaly_Flag:")
for k, v in df['Anomaly_Flag'].value_counts().items():
    pct = v / len(df) * 100
    print(f"      - {k:22s}: {v:,} baris ({pct:.2f}%)")

outliers = df[df['Anomaly_Flag'] == 'Outlier Gross']
print()
print(f"      Detail Transaksi Outlier Gross (>= Rp500.000):")
for _, r in outliers[['Receipt Number', 'Datetime', 'Items', 'Quantity', 'Gross Sales', 'Discounts', 'Net Sales']].iterrows():
    print(f"      * Struk {r['Receipt Number']} ({r['Datetime'].strftime('%Y-%m-%d %H:%M:%S')}): {r['Quantity']}x {r['Items']} | Gross: Rp {r['Gross Sales']:,.0f} | Disc: Rp {r['Discounts']:,.0f} | Net: Rp {r['Net Sales']:,.0f}")


[5.2] Distribusi Kolom Anomaly_Flag:
      - Normal                : 58,125 baris (94.62%)
      - Gratis/Diskon Penuh   : 3,298 baris (5.37%)
      - Kemungkinan Uji Coba  : 7 baris (0.01%)
      - Outlier Gross         : 1 baris (0.00%)

      Detail Transaksi Outlier Gross (>= Rp500.000):
      * Struk 3BOOCW (2026-03-04 22:55:04): 50x Mie Gebres | Gross: Rp 500,000 | Disc: Rp 500,000 | Net: Rp 0


### 5.3 Penandaan Kategori Khusus (`Item_Flag`)
Menandai menu khusus ke kolom `Item_Flag`: `"Uncategorized"` sebagai `"Custom Amount"`, `"Bundle_Package"` sebagai `"Bundle"`, dan kategori lainnya sebagai `"Normal"`.

In [14]:
# 5.3 Menambahkan kolom Item_Flag berdasarkan Category
print(f"[5.3] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

def assign_item_flag(category):
    if category == 'Uncategorized':
        return 'Custom Amount'
    elif category == 'Bundle_Package':
        return 'Bundle'
    else:
        return 'Normal'

df['Item_Flag'] = df['Category'].apply(assign_item_flag)

print(f"[5.3] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah 'Item_Flag')")
print("      Distribusi Item_Flag:")
for k, v in df['Item_Flag'].value_counts().items():
    pct = v / len(df) * 100
    print(f"      - {k:15s}: {v:,} baris ({pct:.2f}%)")

print("\n      3 Item teratas per kategori khusus:")
for cat_name in ['Bundle_Package', 'Uncategorized']:
    sub_df = df[df['Category'] == cat_name]
    print(f"      * Kategori {cat_name}:")
    for itm, cnt in sub_df['Items'].value_counts().head(3).items():
        print(f"        - {itm}: {cnt} baris")


[5.3] SEBELUM : 61,431 baris | 23 kolom
[5.3] SESUDAH : 61,431 baris | 24 kolom (tambah 'Item_Flag')
      Distribusi Item_Flag:
      - Normal         : 61,317 baris (99.81%)
      - Bundle         : 80 baris (0.13%)
      - Custom Amount  : 34 baris (0.06%)

      3 Item teratas per kategori khusus:
      * Kategori Bundle_Package:
        - Mie Ayam Juara: 11 baris
        - Mie Ayam Bakso Mantap: 11 baris
        - Paket Bakso Klasik Kecil: 8 baris
      * Kategori Uncategorized:
        - Custom Amount: 34 baris


## 6. Penyeragaman Label & Parsing Data Multi-Value


### 6.1 Standarisasi Alasan Refund (`Reason of Refund` & `Reason_Raw`)
Memetakan alasan refund yang bervariasi ke 8 kategori baku, serta menyimpan teks asli di kolom `Reason_Raw`.


In [15]:
# 6.1 Standarisasi alasan refund
print(f"[6.1] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

# Salin teks asli Reason of Refund, isi NaN dengan 'Tidak Ada'
df['Reason_Raw'] = df['Reason of Refund'].fillna('Tidak Ada')

def map_refund_reason(reason):
    if pd.isna(reason) or reason in ['None', 'Tidak Ada'] or str(reason).strip() == '':
        return 'Tidak Ada'
    r = str(reason).lower().strip()
    if 'trial' in r or 'belajar' in r:
        return 'Trial/Training'
    elif 'dobel' in r or 'double' in r:
        return 'Double Input'
    elif 'belum ready' in r or 'blm ready' in r:
        return 'Item Not Ready'
    elif 'poin' in r or 'cash' in r:
        return 'Customer Request (Points/Cash)'
    elif 'ganti' in r or 'pergantian' in r:
        return 'Product Replacement'
    elif 'salah' in r or 'accidental' in r or 'salah ketik' in r:
        return 'Accidental Charge'
    elif 'cancel' in r or 'batal' in r:
        return 'Cancelled Order'
    else:
        return 'Other'

df['Reason of Refund'] = df['Reason_Raw'].apply(map_refund_reason)

print(f"[6.1] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah 'Reason_Raw' & standarisasi 'Reason of Refund')")
print(f"      Distribusi Kategori Alasan Refund:")
for k, v in df[df['Is_Refund']]['Reason of Refund'].value_counts().items():
    print(f"      - {k:32s}: {v:,} baris")


[6.1] SEBELUM : 61,431 baris | 24 kolom
[6.1] SESUDAH : 61,431 baris | 25 kolom (tambah 'Reason_Raw' & standarisasi 'Reason of Refund')
      Distribusi Kategori Alasan Refund:
      - Cancelled Order                 : 59 baris
      - Accidental Charge               : 33 baris
      - Customer Request (Points/Cash)  : 28 baris
      - Trial/Training                  : 7 baris
      - Double Input                    : 6 baris
      - Item Not Ready                  : 2 baris
      - Product Replacement             : 2 baris


### 6.2 Standarisasi Metode Pembayaran (`Payment Method`)
Menyeragamkan penamaan metode split payment (`Gopay;Cash` $
ightarrow$ `Split (Gopay+Cash)`).


In [16]:
# 6.2 Menyeragamkan label Payment Method
print(f"[6.2] Nilai Payment Method sebelum standarisasi:")
print(f"      Gopay;Cash muncul sebanyak : {(df['Payment Method'] == 'Gopay;Cash').sum():,} baris")

df['Payment Method'] = df['Payment Method'].replace({'Gopay;Cash': 'Split (Gopay+Cash)'})

print(f"      Nilai Payment Method sesudah standarisasi:")
print(f"      Split (Gopay+Cash) muncul  : {(df['Payment Method'] == 'Split (Gopay+Cash)').sum():,} baris")


[6.2] Nilai Payment Method sebelum standarisasi:
      Gopay;Cash muncul sebanyak : 33 baris
      Nilai Payment Method sesudah standarisasi:
      Split (Gopay+Cash) muncul  : 33 baris


### 6.3 Pemeriksaan Kanal Khusus `Sales Type` (`QPON` & `ONLINE`)
Mencatat baris transaksi kanal unik yang sangat jarang muncul agar terdokumentasi dengan transparan.


In [17]:
# 6.3 Log baris QPON dan ONLINE
special_sales = df[df['Sales Type'].isin(['QPON', 'ONLINE'])]
print(f"[6.3] Baris transaksi dengan Sales Type khusus:")
for _, r in special_sales[['Receipt Number', 'Datetime', 'Sales Type', 'Items', 'Quantity', 'Net Sales']].iterrows():
    print(f"      - Struk {r['Receipt Number']} [{r['Sales Type']}]: {r['Quantity']}x {r['Items']} (Net: Rp {r['Net Sales']:,.0f})")


[6.3] Baris transaksi dengan Sales Type khusus:
      - Struk 3BOML6 [ONLINE]: 1x Bakso Urat Jumbo (Net: Rp 15,000)
      - Struk 3BOOEI [QPON]: 3x Bakso Goreng (Net: Rp 10,714)


### 6.4 Parsing Multi-Value `Modifier Applied` Menjadi `Modifier_List`
Mempertahankan kolom asli `Modifier Applied` dan membuat kolom `Modifier_List` berisi string elemen dipisahkan pipa (`|`).


In [18]:
# 6.4 Parsing Modifier Applied ke Modifier_List
print(f"[6.4] SEBELUM : {df.shape[0]:,} baris | {df.shape[1]} kolom")

def parse_modifier(val):
    if pd.isna(val) or val in ['None', 'Tidak Ada'] or str(val).strip() == '':
        return 'Tidak Ada'
    parts = [p.strip() for p in str(val).split(',') if p.strip()]
    return '|'.join(parts) if parts else 'Tidak Ada'

df['Modifier_List'] = df['Modifier Applied'].apply(parse_modifier)
df['Modifier Applied'] = df['Modifier Applied'].fillna('Tidak Ada')

print(f"[6.4] SESUDAH : {df.shape[0]:,} baris | {df.shape[1]} kolom (tambah kolom 'Modifier_List')")
print(f"      Contoh Modifier_List: {df[df['Modifier_List'] != 'Tidak Ada']['Modifier_List'].iloc[0]}")


[6.4] SEBELUM : 61,431 baris | 25 kolom
[6.4] SESUDAH : 61,431 baris | 26 kolom (tambah kolom 'Modifier_List')
      Contoh Modifier_List: Level Pedas - Level 0


### 6.5 Standarisasi Nilai Kosong pada `Discount Applied`
Mengisi missing value pada kolom `Discount Applied` dengan nilai baku `'Tidak Ada'`.

In [19]:
# 6.5 Mengisi null Discount Applied dengan Tidak Ada
null_before = df['Discount Applied'].isna().sum()
df['Discount Applied'] = df['Discount Applied'].fillna('Tidak Ada')
null_after = df['Discount Applied'].isna().sum()

print(f"[6.5] Standarisasi Discount Applied: {null_before:,} nilai kosong diisi dengan 'Tidak Ada'. Sisa null = {null_after}")


[6.5] Standarisasi Discount Applied: 54,350 nilai kosong diisi dengan 'Tidak Ada'. Sisa null = 0


## 7. Validasi Akhir Mutu Data & Rekonsiliasi Finansial


### 7.1 Validasi Rumus Finansial (`Net Sales == Gross - Discounts - Refunds`)
Menguji konsistensi formula finansial di seluruh baris data menggunakan assertion matematis.


In [20]:
# 7.1 Pengujian formula Net Sales
expected_net = df['Gross Sales'] - df['Discounts'] - df['Refunds']
diff_formula = (df['Net Sales'] - expected_net).abs()
max_diff = diff_formula.max()
invalid_count = (diff_formula > 0.5).sum()

print(f"[7.1] Validasi Formula Net Sales:")
print(f"      - Maksimum selisih pembulatan : Rp {max_diff:.4f}")
print(f"      - Baris tidak konsisten       : {invalid_count} baris")

assert invalid_count == 0, f"Ditemukan {invalid_count} baris yang melanggar formula Net Sales!"
print("      [ASSERT PASS] Seluruh baris 100% konsisten dengan formula pembukuan kasir.")


[7.1] Validasi Formula Net Sales:
      - Maksimum selisih pembulatan : Rp 0.0300
      - Baris tidak konsisten       : 0 baris
      [ASSERT PASS] Seluruh baris 100% konsisten dengan formula pembukuan kasir.


### 7.2 Validasi Kelengkapan Kolom Inti (*Core Columns Completeness*)
Memastikan tidak ada nilai null pada kolom identitas utama dan finansial penting.


In [21]:
# 7.2 Pengecekan missing values kolom inti
core_columns = ['Datetime', 'Category', 'Items', 'Quantity', 'Gross Sales', 'Net Sales']
missing_summary = df[core_columns].isna().sum()

print(f"[7.2] Pemeriksaan Missing Values Kolom Inti:")
for col, missing in missing_summary.items():
    print(f"      - {col:15s}: {missing} null values")

assert missing_summary.sum() == 0, "Ditemukan nilai kosong pada kolom inti!"
print("      [ASSERT PASS] Seluruh kolom inti 100% terisi tanpa ada data hilang.")


[7.2] Pemeriksaan Missing Values Kolom Inti:
      - Datetime       : 0 null values
      - Category       : 0 null values
      - Items          : 0 null values
      - Quantity       : 0 null values
      - Gross Sales    : 0 null values
      - Net Sales      : 0 null values
      [ASSERT PASS] Seluruh kolom inti 100% terisi tanpa ada data hilang.


### 7.3 Pembersihan Whitespace pada Semua Kolom Teks
Menghilangkan spasi liar di awal dan akhir teks pada semua kolom bertipe string object.


In [22]:
# 7.3 Pembersihan whitespace teks
text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())

print(f"[7.3] Berhasil membersihkan leading/trailing whitespace pada {len(text_cols)} kolom teks (mempertahankan NaN asli).")


[7.3] Berhasil membersihkan leading/trailing whitespace pada 15 kolom teks (mempertahankan NaN asli).


### 7.4 Rekonsiliasi Total Finansial Sebelum vs Sesudah Cleaning (Memperhitungkan Dropped Nagih)
Menguji formula rekonsiliasi:
$$\text{Total Awal (Raw)} - \text{Total Dibuang (Nagih)} = \text{Total Bersih Final}$$
Memastikan tidak ada selisih uang yang hilang atau bertambah tanpa alasan.


In [23]:
# 7.4 Rekonsiliasi finansial komprehensif
raw_gross_total = df_raw['Gross Sales'].sum()
raw_net_total = df_raw['Net Sales'].sum()

dropped_gross_total = dropped_nagih['Gross Sales'].sum()
dropped_net_total = dropped_nagih['Net Sales'].sum()

clean_gross_total = df['Gross Sales'].sum()
clean_net_total = df['Net Sales'].sum()

diff_gross_reconcile = abs((raw_gross_total - dropped_gross_total) - clean_gross_total)
diff_net_reconcile = abs((raw_net_total - dropped_net_total) - clean_net_total)

print(f"[7.4] Rekonsiliasi Finansial:")
print(f"      Gross Sales : Raw Rp {raw_gross_total:,.2f} - Dropped Rp {dropped_gross_total:,.2f} == Clean Rp {clean_gross_total:,.2f} | Selisih: Rp {diff_gross_reconcile:.4f}")
print(f"      Net Sales   : Raw Rp {raw_net_total:,.2f} - Dropped Rp {dropped_net_total:,.2f} == Clean Rp {clean_net_total:,.2f} | Selisih: Rp {diff_net_reconcile:.4f}")

assert diff_gross_reconcile < 0.01, f"Gross Sales tidak seimbang! Selisih = {diff_gross_reconcile}"
assert diff_net_reconcile < 0.01, f"Net Sales tidak seimbang! Selisih = {diff_net_reconcile}"
print("      [ASSERT PASS] Rekonsiliasi finansial 100% seimbang sempurna.")

# Sanity-check baseline target (peringatan bila berbeda, bukan assert)
expected_nagih_receipts = 98
expected_dropped_rows = 392
expected_final_rows = 61431
expected_net_sales = 544329042.52
expected_gross_sales = 577677500.00

print(f"\n[7.4 Sanity-Check Baseline Target]:")
print(f"      * Struk Nagih       : {num_nagih_receipts} " + ("[OK]" if num_nagih_receipts == expected_nagih_receipts else f"[PERINGATAN: target {expected_nagih_receipts}]"))
print(f"      * Baris Dibuang     : {len(dropped_nagih)} " + ("[OK]" if len(dropped_nagih) == expected_dropped_rows else f"[PERINGATAN: target {expected_dropped_rows}]"))
print(f"      * Baris Final       : {len(df):,} " + ("[OK]" if len(df) == expected_final_rows else f"[PERINGATAN: target {expected_final_rows:,}]"))
print(f"      * Net Sales Final   : Rp {clean_net_total:,.2f} " + ("[OK]" if abs(clean_net_total - expected_net_sales) < 0.01 else f"[PERINGATAN: target Rp {expected_net_sales:,.2f}]"))
print(f"      * Gross Sales Final : Rp {clean_gross_total:,.2f} " + ("[OK]" if abs(clean_gross_total - expected_gross_sales) < 0.01 else f"[PERINGATAN: target Rp {expected_gross_sales:,.2f}]"))


[7.4] Rekonsiliasi Finansial:
      Gross Sales : Raw Rp 579,298,500.00 - Dropped Rp 1,621,000.00 == Clean Rp 577,677,500.00 | Selisih: Rp 0.0000
      Net Sales   : Raw Rp 545,722,042.59 - Dropped Rp 1,393,000.07 == Clean Rp 544,329,042.52 | Selisih: Rp 0.0000
      [ASSERT PASS] Rekonsiliasi finansial 100% seimbang sempurna.

[7.4 Sanity-Check Baseline Target]:
      * Struk Nagih       : 98 [OK]
      * Baris Dibuang     : 392 [OK]
      * Baris Final       : 61,431 [OK]
      * Net Sales Final   : Rp 544,329,042.52 [OK]
      * Gross Sales Final : Rp 577,677,500.00 [OK]


## 8. Ekspor Output & Pembuatan Laporan


### 8.1 Ekspor Dataset Bersih Final (`sales_clean.csv`)
Mengekspor data final bersih hasil agregasi dan standarisasi ke file `sales_clean.csv`.


In [24]:
# 8.1 Simpan file sales_clean.csv
output_clean = 'sales_clean.csv'
df.to_csv(output_clean, index=False)

print(f"[8.1] File '{output_clean}' berhasil disimpan: {df.shape[0]:,} baris x {df.shape[1]} kolom")

# Pemeriksaan ketat: tidak boleh ada sel berisi teks 'nan' atau 'None'
df_clean_check = pd.read_csv(output_clean, keep_default_na=False)
for col in df_clean_check.columns:
    bad_nan = (df_clean_check[col] == 'nan').sum()
    bad_none = (df_clean_check[col] == 'None').sum()
    assert bad_nan == 0, f"Kolom '{col}' di {output_clean} memuat {bad_nan} sel teks 'nan'!"
    assert bad_none == 0, f"Kolom '{col}' di {output_clean} memuat {bad_none} sel teks 'None'!"

print(f"      [ASSERT PASS] Tidak ada sel bernilai teks 'nan' atau 'None' di {output_clean}.")

# Validasi ketat bahwa kolom Customer telah dihapus dari seluruh berkas output utama
for f in ['sales_clean.csv', 'refunds.csv', 'dropped_nagih.csv']:
    assert 'Customer' not in pd.read_csv(f, nrows=0).columns, f"Kolom Customer masih ada di {f}"
print("      [ASSERT PASS] Kolom Customer terbukti tidak ada di sales_clean.csv, refunds.csv, dan dropped_nagih.csv.")


[8.1] File 'sales_clean.csv' berhasil disimpan: 61,431 baris x 26 kolom
      [ASSERT PASS] Tidak ada sel bernilai teks 'nan' atau 'None' di sales_clean.csv.
      [ASSERT PASS] Kolom Customer terbukti tidak ada di sales_clean.csv, refunds.csv, dan dropped_nagih.csv.


In [25]:
df.head(5)

,Receipt Number,Category,Items,Modifier Applied,Discount Applied,Sales Type,Payment Method,Event Type,Reason of Refund,Datetime,...,Quantity,Gross Sales,Discounts,Refunds,Net Sales,Merged_Row_Count,Anomaly_Flag,Item_Flag,Reason_Raw,Modifier_List
0,3BOM8J,Mie,Mie Barbeque,Tidak Ada,Tidak Ada,DINE IN,Gopay,Payment,Tidak Ada,2026-01-01 10:49:30,...,1,20000.0,0.0,0.0,20000.0,1,Normal,Normal,Tidak Ada,Tidak Ada
1,3BOM8K,Bakso,Bakso Isi Keju,Tidak Ada,Tidak Ada,DINE IN,Cash,Payment,Tidak Ada,2026-01-01 10:52:41,...,1,8000.0,0.0,0.0,8000.0,1,Normal,Normal,Tidak Ada,Tidak Ada
2,3BOM8K,Bakso,Bakso Jamur,Tidak Ada,Tidak Ada,DINE IN,Cash,Payment,Tidak Ada,2026-01-01 10:52:41,...,1,8000.0,0.0,0.0,8000.0,1,Normal,Normal,Tidak Ada,Tidak Ada
3,3BOM8K,Bakso,Bakso Jumbo Merekah,Tidak Ada,Tidak Ada,DINE IN,Cash,Payment,Tidak Ada,2026-01-01 10:52:41,...,1,18000.0,0.0,0.0,18000.0,1,Normal,Normal,Tidak Ada,Tidak Ada
4,3BOM8K,Bakso,Bakso Kecil,Tidak Ada,Tidak Ada,DINE IN,Cash,Payment,Tidak Ada,2026-01-01 10:52:41,...,2,6000.0,0.0,0.0,6000.0,1,Normal,Normal,Tidak Ada,Tidak Ada


### 8.2 Ekspor Data Refund Lengkap (`refunds.csv`)
Menyimpan file `refunds.csv` lengkap berisi 149 transaksi refund murni sebelum agregasi duplikat agar riwayat struk refund tetap utuh untuk audit kasir.


In [26]:
# 8.2 Simpan file refunds.csv lengkap
output_refund = 'refunds.csv'

# Standardisasi label pada data refund audit
df_refunds['Discount Applied'] = df_refunds['Discount Applied'].fillna('Tidak Ada')
df_refunds['Anomaly_Flag'] = df_refunds.apply(detect_anomaly, axis=1)
df_refunds['Reason_Raw'] = df_refunds['Reason of Refund'].fillna('Tidak Ada')
df_refunds['Reason of Refund'] = df_refunds['Reason_Raw'].apply(map_refund_reason)
df_refunds['Modifier_List'] = df_refunds['Modifier Applied'].apply(parse_modifier)
df_refunds['Modifier Applied'] = df_refunds['Modifier Applied'].fillna('Tidak Ada')
df_refunds['Payment Method'] = df_refunds['Payment Method'].replace({'Gopay;Cash': 'Split (Gopay+Cash)'})
df_refunds['Item_Flag'] = df_refunds['Category'].apply(assign_item_flag)

# Drop kolom Date dan Time dari refunds jika ada
df_refunds = df_refunds.drop(columns=[c for c in ['Date', 'Time'] if c in df_refunds.columns])

# Pembersihan whitespace teks pada kolom object
for col in df_refunds.select_dtypes(include='object').columns:
    df_refunds[col] = df_refunds[col].where(df_refunds[col].isna(), df_refunds[col].astype(str).str.strip())

df_refunds.to_csv(output_refund, index=False)
print(f"[8.2] File '{output_refund}' berhasil disimpan: {df_refunds.shape[0]:,} baris x {df_refunds.shape[1]} kolom")

# Pemeriksaan ketat: tidak boleh ada sel berisi teks 'nan' atau 'None'
df_ref_check = pd.read_csv(output_refund, keep_default_na=False)
for col in df_ref_check.columns:
    bad_nan = (df_ref_check[col] == 'nan').sum()
    bad_none = (df_ref_check[col] == 'None').sum()
    assert bad_nan == 0, f"Kolom '{col}' di {output_refund} memuat {bad_nan} sel teks 'nan'!"
    assert bad_none == 0, f"Kolom '{col}' di {output_refund} memuat {bad_none} sel teks 'None'!"

print(f"      [ASSERT PASS] Tidak ada sel bernilai teks 'nan' atau 'None' di {output_refund}.")


[8.2] File 'refunds.csv' berhasil disimpan: 149 baris x 25 kolom
      [ASSERT PASS] Tidak ada sel bernilai teks 'nan' atau 'None' di refunds.csv.


### 8.3 Penyusunan & Ekspor Dokumen Laporan (`cleaning_report.md`)
Menghasilkan dokumen audit komprehensif yang merangkum hasil pembersihan, keputusan penghapusan kolom redundan, serta daftar pertanyaan terbuka.


In [27]:
# 8.3 Membuat dokumen laporan cleaning_report.md
report_file = 'cleaning_report.md'

def series_to_md(s, label):
    lines = [f'| {label} | Jumlah Baris | Persentase |', '|---|---|---|']
    total = s.sum()
    for k, v in s.items():
        pct = (v / total * 100)
        lines.append(f'| `{k}` | {v:,} | {pct:.2f}% |')
    return '\n'.join(lines)

table_category = series_to_md(df['Category'].value_counts(), 'Kategori Menu (Category)')
table_anomaly_flag = series_to_md(df['Anomaly_Flag'].value_counts(), 'Jenis Anomali (Anomaly_Flag)')
table_item_flag = series_to_md(df['Item_Flag'].value_counts(), 'Penanda Menu (Item_Flag)')

payments_clean = df[df['Event Type'] == 'Payment']
refunds_clean = df[df['Event Type'] == 'Refund']
raw_payments = df_raw[df_raw['Event Type'] == 'Payment']
raw_refunds = df_raw[df_raw['Event Type'] == 'Refund']
dropped_payments = dropped_nagih[dropped_nagih['Event Type'] == 'Payment']
dropped_refunds = dropped_nagih[dropped_nagih['Event Type'] == 'Refund']

# Hitungan dinamis kolom
cols_dropped_count = len(drop_cols)
cols_dropped_list_str = ', '.join([f'`{c}`' for c in drop_cols])
new_cols_added = [c for c in df.columns if c not in df_raw.columns]
new_cols_count = len(new_cols_added)
new_cols_list_str = ', '.join([f'`{c}`' for c in new_cols_added])

# Hitungan dinamis Nagih
dropped_nagih_total = len(dropped_nagih)
dropped_nagih_only_count = (dropped_nagih['Category'] == '0-Nagih').sum()
dropped_non_nagih_count = dropped_nagih_total - dropped_nagih_only_count

# Hitungan dinamis Refund
refund_raw_count = len(df_refunds)
refund_clean_count = len(refunds_clean)

# Hitungan dinamis Anomali
outlier_gross_count = (df['Anomaly_Flag'] == 'Outlier Gross').sum()
trial_count = (df['Anomaly_Flag'] == 'Kemungkinan Uji Coba').sum()
gratis_count = (df['Anomaly_Flag'] == 'Gratis/Diskon Penuh').sum()
normal_count = (df['Anomaly_Flag'] == 'Normal').sum()

# Hitungan dinamis Kategori Khusus (Item_Flag)
bundle_count = (df['Category'] == 'Bundle_Package').sum()
uncat_count = (df['Category'] == 'Uncategorized').sum()
normal_item_flag_count = (df['Item_Flag'] == 'Normal').sum()
uncat_net = df[df['Category'] == 'Uncategorized']['Net Sales'].sum()
top_bundle_items = df[df['Category'] == 'Bundle_Package']['Items'].value_counts().head(3)
top_bundle_str = ', '.join([f'{itm} ({cnt} baris)' for itm, cnt in top_bundle_items.items()])

main_cats = ['Bakso', 'Gorengan', 'Mie', 'Minuman', 'Tambahan']
main_cat_count = df[df['Category'].isin(main_cats)].shape[0]
main_cat_pct = main_cat_count / len(df) * 100

# Data dinamis struk 3BOOCW
row_3b = df[df['Receipt Number'] == '3BOOCW'].iloc[0]
indo_months = {1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April', 5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus', 9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'}
date_3b_str = f"{row_3b['Datetime'].day} {indo_months[row_3b['Datetime'].month]} {row_3b['Datetime'].year}"
item_3b_str = row_3b['Items']
qty_3b = int(row_3b['Quantity'])
gross_3b = row_3b['Gross Sales']
disc_3b = row_3b['Discounts']
net_3b = row_3b['Net Sales']

# Data dinamis struk 4Z7 (dihitung dari df_raw sebelum Collected By dibuang)
df_4z7 = df_raw[df_raw['Receipt Number'].str.startswith('4Z7')]
count_4z7_receipts = df_4z7['Receipt Number'].nunique()
count_4z7_rows = len(df_4z7)
gross_4z7_total = df_4z7['Gross Sales'].sum()
net_4z7_total = df_4z7['Net Sales'].sum()
kasir_4z7_parts = []
for kasir, grp in df_4z7.groupby('Collected By'):
    kasir_4z7_parts.append(f"kasir `{kasir}` ({len(grp)} baris, {grp['Receipt Number'].nunique()} struk, Gross: Rp {grp['Gross Sales'].sum():,.2f}, Net: Rp {grp['Net Sales'].sum():,.2f})")
kasir_4z7_desc = '; '.join(kasir_4z7_parts)

report_md = f'''# Laporan Audit & Dokumentasi Data Cleaning Penjualan Bakso Lintang

**Kasus Bisnis:** Warung Bakso Lintang (Ketintang, Surabaya)  
**Dokumen Acuan:** [PRD.md](PRD.md) v1.1 - Sistem Prediksi Stok & Business Intelligence  
**File Input Mentah:** `{input_file}` ({df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom)  
**File Output Bersih:** `{output_clean}` ({df.shape[0]:,} baris x {df.shape[1]} kolom)  
**File Output Refund:** `{output_refund}` ({df_refunds.shape[0]:,} baris)  
**File Arsip Dropped Nagih:** `{file_dropped_nagih}` ({len(dropped_nagih):,} baris)  
**Tanggal Eksekusi:** {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}  

---

## 1. Ringkasan Sebelum vs Sesudah Cleaning

| Metrik | Data Awal (Raw) | Dikeluarkan (Nagih) | Data Bersih (Final) | Status Rekonsiliasi |
|---|---|---|---|---|
| **Jumlah Baris** | {df_raw.shape[0]:,} | {len(dropped_nagih):,} | {df.shape[0]:,} | -{df_raw.shape[0] - df.shape[0]:,} baris ({len(dropped_nagih):,} Nagih + {num_excess_group:,} duplikat) |
| **Jumlah Kolom** | {df_raw.shape[1]} | - | {df.shape[1]} | {df.shape[1] - df_raw.shape[1]:+d} kolom ({cols_dropped_count} kolom dibuang, {new_cols_count} fitur baru/standar) |
| **Total Gross Sales** | Rp {raw_gross_total:,.2f} | Rp {dropped_gross_total:,.2f} | Rp {clean_gross_total:,.2f} | **SAMA PERSIS (Selisih Rp 0.00)** |
| **Total Net Sales** | Rp {raw_net_total:,.2f} | Rp {dropped_net_total:,.2f} | Rp {clean_net_total:,.2f} | **SAMA PERSIS (Selisih Rp 0.00)** |
| **Net Sales (Payment)** | Rp {raw_payments['Net Sales'].sum():,.2f} | Rp {dropped_payments['Net Sales'].sum():,.2f} | Rp {payments_clean['Net Sales'].sum():,.2f} | **SAMA PERSIS (Selisih Rp 0.00)** |
| **Net Sales (Refund)** | Rp {raw_refunds['Net Sales'].sum():,.2f} | Rp {dropped_refunds['Net Sales'].sum():,.2f} | Rp {refunds_clean['Net Sales'].sum():,.2f} | **SAMA PERSIS (Selisih Rp 0.00)** |

---

## 2. Matriks Penanganan Data Cleaning (Kondisi Awal, Penanganan, Alasan, dan Hasil)

Berikut adalah dokumentasi terstruktur untuk setiap isu kualitas data yang ditemukan:

### A. Penyatuan dan Standarisasi Tanggal & Waktu
- **Kondisi Awal:** Kolom tanggal (`Date`) dan jam (`Time`) terpisah sebagai string dengan format lokal (`DD-MM-YYYY` dan `HH:MM:SS`), belum bertipe datetime presisi.
- **Penanganan:** Menggabungkan keduanya menjadi satu kolom tunggal **`Datetime`** bertipe `datetime64[ns]`, kemudian mengekstrak 4 atribut waktu analitis (`Year_Month`, `Day_Name`, `Hour`, `Is_Weekend`). Kolom `Date` dan `Time` mentah dilepas.
- **Alasan:** Format timestamp presisi dibutuhkan sebagai dasar urutan deret waktu pada model peramalan kebutuhan bahan baku dapur ([PRD.md](PRD.md) FR-04), dan mempermudah analisis pola operasional jam sibuk tanpa redundansi data.
- **Hasil:** 1 kolom timestamp presisi (`Datetime`) dan 4 fitur waktu siap pakai untuk eksplorasi tren bisnis.

### B. Transaksi Kategori "0-Nagih" (Uji Coba Penjualan Warkop)
- **Kondisi Awal:** Terdapat {num_nagih_receipts:,} nomor struk kasir yang memuat item berkategori `0-Nagih` ({dropped_nagih_only_count:,} baris menu Nagih seperti '0-Somay', '0-Kecil', dan {dropped_non_nagih_count:,} baris menu bakso reguler yang tercatat pada struk yang sama).
- **Penanganan:** Mengidentifikasi seluruh `Receipt Number` yang mengandung `0-Nagih`, membuang seluruh baris dari struk tersebut, dan menyimpannya ke berkas arsip `{file_dropped_nagih}`.
- **Alasan:** Kategori 0-Nagih adalah uji coba penjualan di warkop (21 Feb-13 Mar 2026) yang tidak dilanjutkan, berdasarkan konfirmasi Owner. Karena bukan penjualan operasional warung, seluruh struk terkait dikeluarkan dan diarsipkan di dropped_nagih.csv. Awalan struk 9XT (kasir Reny Komar) dan 8AA (kasir Insan Admin) adalah bagian dari uji coba ini.
- **Hasil:** Sebanyak {len(dropped_nagih):,} baris ({num_nagih_receipts:,} struk) berhasil dikeluarkan dan diarsipkan lengkap (Gross: Rp {dropped_gross_total:,.2f}, Net: Rp {dropped_net_total:,.2f}). Awalan struk `9XT` (kasir `Reny Komar`) dan `8AA` (kasir `Insan Admin`) otomatis bersih dari data final.

### C. Kolom Tidak Berguna, Redundan, dan Kosong
- **Kondisi Awal:** Terdapat {cols_dropped_count} kolom yang tidak memberikan nilai analitis atau 100% bernilai kosong dari sistem kasir ({cols_dropped_list_str}).
- **Penanganan:** Menghapus seluruh {cols_dropped_count} kolom tersebut dari DataFrame (`df.drop(columns=drop_cols)`).
- **Alasan:** Kolom kosong dan konstanta tidak menambah variansi informasi, sedangkan kolom `Date` dan `Time` telah diwakili secara lebih akurat dan terintegrasi oleh kolom tunggal `Datetime`.
- **Hasil:** Struktur tabel lebih ramping dan efisien, menghemat penggunaan memori tanpa kehilangan informasi analitis penting.

### D. Pemisahan dan Standarisasi Transaksi Refund
- **Kondisi Awal:** Terdapat transaksi refund ({refund_raw_count:,} baris sebelum agregasi) yang tercampur di dalam aliran transaksi pembayaran reguler.
- **Penanganan:** Menandai baris refund (`Is_Refund = True`), mengekstrak tipe refund (`Refund_Type` = 'Full' / 'Partial' / 'Tidak Ada') dan nama item asli, memetakan alasan refund ke kategori baku, serta mengekspor data audit refund ke `{output_refund}` sebelum agregasi.
- **Alasan:** Menjaga riwayat audit kasir tetap utuh dan transparan sebelum baris agregasi disatukan.
- **Hasil:** Berkas `{output_refund}` tersimpan rapi ({refund_raw_count:,} baris, Net Sales: Rp {df_refunds['Net Sales'].sum():,.2f}).
*Catatan: File `refunds.csv` (sebelum agregasi) berisi {refund_raw_count:,} baris, sedangkan baris refund di `sales_clean.csv` (sesudah agregasi) berisi {refund_clean_count:,} baris; perbedaan ini wajar karena adanya penggabungan baris refund dengan atribut identik dalam satu struk.*

### E. Baris dengan Atribut Identik Menggunakan Agregasi (Bukan Drop Mentah)
- **Kondisi Awal:** Ditemukan {num_exact_all_cols:,} baris duplikat persis di semua kolom, dan {num_excess_group:,} baris berlebih dengan atribut identik (kuantitas/nilai boleh berbeda) dalam satu nomor struk (melibatkan total {num_total_involved:,} baris).
- **Penanganan:** Mengelompokkan berdasarkan atribut non-metrik dan menjumlahkan metrik kuantitas serta finansial (`Quantity`, `Gross Sales`, `Discounts`, `Refunds`, `Net Sales`), disertai kolom pelacak `Merged_Row_Count`.
- **Alasan:** Diduga pemesanan bertahap (belum dikonfirmasi Owner), seperti pelanggan yang menambah porsi saat makan tanpa membuka struk baru. Jika dibuang mentah menggunakan `drop_duplicates()`, omzet dan kuantitas penjualan riil akan hilang.
- **Hasil:** Sebanyak {num_excess_group:,} baris berlebih berhasil dirampingkan menjadi baris tunggal teragregasi dengan integritas nominal 100% terjaga (selisih Rp 0,00).

### F. Penandaan Anomali Transaksi (`Anomaly_Flag`)
- **Kondisi Awal:** Terdapat transaksi ekstrem dan transaksi uji coba kasir yang belum memiliki label penanda.
- **Penanganan:** Membuat kolom `Anomaly_Flag` dengan kategori: 'Outlier Gross' (Gross >= Rp 500.000), 'Kemungkinan Uji Coba' (alasan refund trial/belajar), 'Gratis/Diskon Penuh' (Net = 0 pada event Payment), dan 'Normal'. Baris tidak dihapus melainkan hanya ditandai.
- **Alasan:** Memberikan transparansi pada tahap EDA dan permodelan tanpa merusak integritas rekonsiliasi data mentah.
- **Hasil:** Terpetakan {outlier_gross_count:,} baris *Outlier Gross*, {trial_count:,} baris kemungkinan uji coba kasir, {gratis_count:,} baris pesanan gratis/diskon penuh, dan {normal_count:,} baris transaksi berstatus normal.

### G. Standarisasi Teks, Modifier, dan Pembayaran
- **Kondisi Awal:** Kolom `Reason of Refund` memiliki teks bebas tidak seragam, `Modifier Applied` memiliki format multi-value dipisah koma dengan spasi, metode split payment berlabel `Gopay;Cash`, dan terdapat spasi liar. Nilai kosong pada kolom kategorikal belum terstandarisasi.
- **Penanganan:** Memetakan alasan refund ke kategori baku (teks asli disimpan di `Reason_Raw`), memparsing `Modifier Applied` menjadi `Modifier_List` dipisahkan pipa (`|`), menyeragamkan payment method menjadi `Split (Gopay+Cash)`, mengisi nilai kosong pada `Discount Applied` dan `Reason_Raw` dengan label baku `'Tidak Ada'`, serta membersihkan leading/trailing whitespace pada seluruh kolom teks tanpa merusak missing values.
- **Alasan:** Standarisasi label mutlak diperlukan agar proses grouping dan filtering saat EDA tidak terpecah oleh inkonsistensi penulisan dan bebas dari string teks 'nan' atau 'None'.
- **Hasil:** Seluruh kolom kategorikal bersih, seragam, dan bebas spasi liar maupun teks 'nan'/'None'.

### H. Data Identitas Pelanggan (`Customer`)
- **Kondisi Awal:** Kolom `Customer` berisi nomor handphone pelanggan format kasir (`+62...`) dan nilai kosong untuk pelanggan anonim.
- **Penanganan:** Kolom `Customer` dihapus secara total dari seluruh berkas output (`sales_clean.csv`, `refunds.csv`, `dropped_nagih.csv`, serta seluruh file siap-pakai).
- **Alasan:** Menjaga privasi data pelanggan (*data privacy/confidentiality*) dan nomor HP tidak dibutuhkan untuk pemodelan peramalan stok bahan baku maupun business intelligence pada PRD.
- **Hasil:** Tidak ada data identitas atau nomor HP pelanggan di seluruh output dataset bersih final.

### I. Penanda Kategori Menu Khusus (`Item_Flag`)
- **Kondisi Awal:** Menu paket (`Bundle_Package`) dan input manual kasir (`Uncategorized`) belum memiliki label pembeda eksplisit dari menu satuan standar.
- **Penanganan:** Membentuk kolom `Item_Flag` berbasis kolom `Category`: `"Uncategorized"` $\rightarrow$ `"Custom Amount"`, `"Bundle_Package"` $\rightarrow$ `"Bundle"`, dan kategori lainnya $\rightarrow$ `"Normal"`.
- **Alasan:** Memudahkan pemisahan data pada model peramalan bahan baku dapur (menu *Bundle* memerlukan dekomposisi resep multi-item, sedangkan *Custom Amount* tidak memiliki resep fisik).
- **Hasil:** Terpetakan {bundle_count:,} baris *Bundle*, {uncat_count:,} baris *Custom Amount*, dan {normal_item_flag_count:,} baris *Normal*.

---

## 3. Hasil Validasi Mutu Data (Data Quality Check)

1. **Konsistensi Formula Keuangan Kasir:**
   - Pengujian `Net Sales == Gross Sales - Discounts - Refunds` terpenuhi di **100% baris** (toleransi pembulatan <= 0.5).
2. **Kelengkapan Kolom Inti (*Core Completeness*):**
   - Kolom inti (`Datetime`, `Category`, `Items`, `Quantity`, `Gross Sales`, `Net Sales`) memiliki **0 missing values (100% lengkap)**.
3. **Pembersihan Whitespace:**
   - Seluruh kolom teks telah dibersihkan dari spasi liar di awal dan akhir karakter dengan mempertahankan nilai kosong asli.
4. **Bebas dari String 'nan', 'None', dan Nomor Telepon:**
   - Seluruh file ekspor telah divalidasi tidak memuat string teks 'nan', 'None', serta bebas dari data nomor telepon pelanggan.
5. **Rekonsiliasi Keuangan (Zero Financial Loss):**
   - Formula `Total Raw - Total Dropped (Nagih) == Total Bersih Final` terpenuhi secara matematis dengan **selisih Rp 0,0000** baik pada Gross Sales maupun Net Sales.

---

## 4. Distribusi Data Bersih Final

### A. Distribusi Kategori Menu (`Category`)

{table_category}

- Menu kategori utama (`Bakso`, `Gorengan`, `Mie`, `Minuman`, `Tambahan`) mencakup **{main_cat_pct:.2f}%** transaksi, yang akan digunakan sebagai input penyusunan Bill of Materials (BOM) pada model prediksi stok ([PRD.md](PRD.md) FR-04).
- Kategori `Bundle_Package` ({bundle_count:,} baris) merupakan paket makanan (Mie Ayam / Paket Bakso), tidak termasuk minuman (contoh: {top_bundle_str}).
- Kategori `Uncategorized` ({uncat_count:,} baris) merupakan Custom Amount (semua item bernama Custom Amount).

### B. Distribusi Indikator Anomali (`Anomaly_Flag`)

{table_anomaly_flag}

### C. Distribusi Penanda Kategori Menu Khusus (`Item_Flag`)

{table_item_flag}

---

## 5. Pertanyaan Lanjutan yang Perlu Dikonfirmasi Owner

Berikut adalah 4 poin keputusan operasional yang perlu dikonfirmasi pemilik warung (Owner) sebelum melatih model machine learning:

1. **Status Omzet Menu "Uncategorized" (Custom Amount):**
   - *Fakta:* Terdapat {uncat_count:,} transaksi manual kasir dengan total Net Sales Rp {uncat_net:,.2f} tanpa keterangan menu fisik (semua item bernama Custom Amount).
   - *Rekomendasi:* Dikecualikan dari model peramalan bahan baku dapur (karena tidak ada resep fisik), tetapi tetap dihitung dalam laporan total omzet kasir.

2. **Pola Pesanan Duplikat dalam Satu Nomor Struk:**
   - *Fakta:* Ditemukan {num_excess_group:,} baris menu berlebih dengan atribut identik pada struk yang sama dan telah diagregasikan kuantitasnya (melibatkan total {num_total_involved:,} baris).
   - *Rekomendasi:* Mohon konfirmasi apakah pola ini diduga pemesanan bertahap (*tambah porsi saat makan*) agar pendekatan agregasi yang diterapkan dinyatakan final.

3. **Validitas Transaksi Pesanan Porsi Besar (Gross Rp 500.000):**
   - *Fakta:* Struk `{row_3b['Receipt Number']}` pada tanggal {date_3b_str} mencatat pesanan {qty_3b} porsi {item_3b_str} senilai Rp {gross_3b:,.2f} dengan diskon Rp {disc_3b:,.2f} sehingga Net Sales = Rp {net_3b:,.2f}.
   - *Rekomendasi:* Transaksi ini kemungkinan konsumsi {qty_3b} porsi; perlu konfirmasi Owner. Selain itu, transaksi ini perlu dikeluarkan saat menghitung *Average Order Value* (AOV) reguler agar tidak mendistorsi statistik keranjang belanja.

4. **Makna Operasional Struk Berawalan "4Z7" (Kasir Nawayoga & Bakso Mambal):**
   - *Fakta:* Terdapat {count_4z7_receipts} struk ({count_4z7_rows} baris) berawalan `4Z7` dengan total Gross Sales Rp {gross_4z7_total:,.2f} dan Net Sales Rp {net_4z7_total:,.2f}, terdiri dari: {kasir_4z7_desc}.
   - *Rekomendasi:* Mohon konfirmasi apakah transaksi ini merupakan pesanan operasional cabang lain, konsumsi internal, atau uji coba kasir baru, agar dapat ditentukan perlakuan pemodelannya.

---

## 6. Berkas Output Siap-Pakai (Ready-to-Use Outputs)

Dataset bersih final diturunkan menjadi 3 file spesifik untuk mendukung kebutuhan operasional downstream:
1. **`sales_final.csv`** ({df.shape[0]:,} baris x 24 kolom): Dataset transaksi granular terstandarisasi snake_case dengan `row_id` sekuensial (1..N) untuk dashboard visualisasi dan chatbot Supabase.
2. **`sales_audit.csv`** ({df.shape[0]:,} baris x 9 kolom): Dataset audit internal untuk menelusuri nomor struk, baris teragregasi (`merged_row_count`), alasan refund mentah, anomali, dan selisih formula keuangan (`net_diff`).
3. **`daily_item_demand.csv`** (18.423 baris: 267 hari x 69 menu): Dataset deret waktu harian per menu untuk peramalan permintaan (*forecasting*) dan penyusunan Bill of Materials (BOM), mencakup pemisahan 264 hari buka dan 3 hari tutup, porsi reguler, porsi refund pengurangan bahan, porsi gratis, serta penanda menu jarang terjual (`is_sparse_item`).
'''

with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report_md)

print(f"[8.3] Dokumen laporan audit '{report_file}' berhasil disusun dan disimpan!")
print()
print("SELURUH PROSES PIPELINE DATA CLEANING TELAH BERHASIL DIJALANKAN DENGAN SEMPURNA!")


[8.3] Dokumen laporan audit 'cleaning_report.md' berhasil disusun dan disimpan!

SELURUH PROSES PIPELINE DATA CLEANING TELAH BERHASIL DIJALANKAN DENGAN SEMPURNA!


## 9. Output Siap-Pakai (Ready-to-Use Analytical Outputs)

Bagian ini memproduksi 3 file output turunan siap-pakai dari DataFrame bersih `df` tanpa mengubah, mengurutkan ulang, atau me-reset indeks `df`, sehingga urutan baris tetap identik 100% dengan `sales_clean.csv`:

1. **`sales_final.csv` (Dashboard & Chatbot Supabase)**:
   - **Tujuan**: Dataset transaksi granular untuk ingestion tabel database Supabase, visualisasi dashboard BI, dan retrieval chatbot AI.
   - **Dimensi**: 61.431 baris x 24 kolom terstandarisasi snake_case.
   - **Struktur Kolom (24 kolom urut)**:
     `row_id` (1..N), `receipt_number`, `datetime` (YYYY-MM-DD HH:MM:SS), `year_month`, `day_name`, `hour`, `is_weekend`, `category`, `item_name`, `modifier_list`, `discount_applied`, `sales_type`, `payment_method`, `event_type`, `is_refund`, `refund_type`, `reason_of_refund`, `quantity`, `gross_sales`, `discounts`, `refunds`, `net_sales`, `anomaly_flag`, `item_flag`.
   - **Pembersihan**: Kolom teknis/audit internal (`Reason_Raw`, `Merged_Row_Count`, `Modifier Applied`) dikecualikan.

2. **`sales_audit.csv` (Penelusuran Audit & Rekonsiliasi)**:
   - **Tujuan**: Penelusuran baris teragregasi, alasan refund mentah, dan validasi selisih formula per baris.
   - **Dimensi**: 61.431 baris x 9 kolom.
   - **Struktur Kolom (9 kolom urut)**:
     `row_id` (identik dengan `sales_final`), `receipt_number`, `datetime`, `item_name`, `merged_row_count`, `reason_raw`, `anomaly_flag`, `item_flag`, `net_diff` ($= \text{round}(\text{Gross} - \text{Discounts} - \text{Refunds} - \text{Net Sales}, 4)$).

3. **`daily_item_demand.csv` (Forecasting Permintaan & BOM Dapur)**:
   - **Tujuan**: Input utama model peramalan deret waktu konsumsi bahan baku dapur dan agregasi Bill of Materials (BOM).
   - **Dimensi**: 18.423 baris (267 hari kalender rentang data x 69 menu terpakai).
   - **Aturan Filter Data Terpakai**:
     - Mengecualikan `Item_Flag` $\in$ `['Custom Amount', 'Bundle']` (tidak memiliki resep tunggal).
     - Mengecualikan `Anomaly_Flag` $\in$ `['Kemungkinan Uji Coba', 'Outlier Gross']` (transaksi non-operasional dapur).
   - **Aturan Agregasi Metrik**:
     - `qty_payment`: Penjualan event `Payment`.
     - `qty_refund`: Pengurangan bahan akibat refund dengan alasan tertentu (`Cancelled Order`, `Accidental Charge`, `Double Input`, `Item Not Ready`).
     - `qty_net`: Target peramalan ($= \text{qty\_payment} + \text{qty\_refund}$).
     - `qty_gratis`: Porsi dari event Payment berdiskon 100% (tetap menghabiskan bahan dapur).
     - `n_receipts`: Jumlah struk unik per menu per hari.
     - `net_sales`: Total omzet bersih per menu per hari.
   - **Penanganan Hari Operasional**:
     - **Hari BUKA (264 hari)**: Kombinasi menu tanpa penjualan diisi $0$, `is_open = True`.
     - **Hari TUTUP (3 hari: 2026-03-22, 2026-03-23, 2026-05-27)**: Warung libur/tutup, seluruh metrik kuantitas dan penjualan DIKOSONGKAN (`NaN`), `is_open = False`.
     - `is_sparse_item`: Bernilai `True` jika total porsi menu sepanjang data $< 30$ porsi (9 menu teridentifikasi).

### Keputusan Bisnis yang Menunggu Konfirmasi Owner:
1. **Alasan Refund Khusus ("Customer Request (Points/Cash)" dan "Product Replacement")**: Saat ini belum mengurangi stok bahan dapur di `qty_refund` karena diduga produk pengganti sudah dikeluarkan atau penyesuaian kasir non-fisik; menunggu konfirmasi fisik dari Owner.
2. **Status Hari Tutup (3 Tanggal)**: Tiga tanggal kosong penjualan (22-23 Maret 2026 awal Ramadhan dan 27 Mei 2026 Idul Adha) dicatat sebagai `is_open = False`.
3. **Struk Berawalan 4Z7**: Net Sales Rp 27.000 (Kasir Nawayoga & Bakso Mambal) tetap dimasukkan ke agregasi menunggu kepastian apakah cabang lain atau internal.
4. **Transaksi 50 Porsi Mie Gebres Struk 3BOOCW**: Ditandai `Outlier Gross` dan dikecualikan dari `daily_item_demand` agar tidak membiaskan model stok reguler harian.

### 9.1 Ekspor `sales_final.csv` (Dashboard & Chatbot Supabase)
Mengekspor 61.431 baris transaksi dengan 24 kolom snake_case dan `row_id` sekuensial (1..N).

In [28]:
# 9.1 Ekspor sales_final.csv
output_sales_final = 'sales_final.csv'

sales_final = pd.DataFrame()
sales_final['row_id'] = np.arange(1, len(df) + 1)
sales_final['receipt_number'] = df['Receipt Number'].values
sales_final['datetime'] = df['Datetime'].dt.strftime('%Y-%m-%d %H:%M:%S').values
sales_final['year_month'] = df['Year_Month'].astype(str).values
sales_final['day_name'] = df['Day_Name'].values
sales_final['hour'] = df['Hour'].values
sales_final['is_weekend'] = df['Is_Weekend'].values
sales_final['category'] = df['Category'].values
sales_final['item_name'] = df['Items'].values
sales_final['modifier_list'] = df['Modifier_List'].values
sales_final['discount_applied'] = df['Discount Applied'].values
sales_final['sales_type'] = df['Sales Type'].values
sales_final['payment_method'] = df['Payment Method'].values
sales_final['event_type'] = df['Event Type'].values
sales_final['is_refund'] = df['Is_Refund'].values
sales_final['refund_type'] = df['Refund_Type'].values
sales_final['reason_of_refund'] = df['Reason of Refund'].values
sales_final['quantity'] = df['Quantity'].values
sales_final['gross_sales'] = df['Gross Sales'].values
sales_final['discounts'] = df['Discounts'].values
sales_final['refunds'] = df['Refunds'].values
sales_final['net_sales'] = df['Net Sales'].values
sales_final['anomaly_flag'] = df['Anomaly_Flag'].values
sales_final['item_flag'] = df['Item_Flag'].values

sales_final_cols = [
    'row_id', 'receipt_number', 'datetime', 'year_month', 'day_name', 'hour', 'is_weekend',
    'category', 'item_name', 'modifier_list', 'discount_applied', 'sales_type', 'payment_method',
    'event_type', 'is_refund', 'refund_type', 'reason_of_refund', 'quantity', 'gross_sales',
    'discounts', 'refunds', 'net_sales', 'anomaly_flag', 'item_flag'
]
sales_final = sales_final[sales_final_cols]
sales_final.to_csv(output_sales_final, index=False)

print(f"[9.1] File '{output_sales_final}' berhasil disimpan: {sales_final.shape[0]:,} baris x {sales_final.shape[1]} kolom")


[9.1] File 'sales_final.csv' berhasil disimpan: 61,431 baris x 24 kolom


### 9.2 Ekspor `sales_audit.csv` (Penelusuran Audit)
Mengekspor 61.431 baris audit yang memuat `row_id`, nomor struk, `merged_row_count`, `reason_raw`, dan `net_diff`.

In [29]:
# 9.2 Ekspor sales_audit.csv
output_sales_audit = 'sales_audit.csv'

sales_audit = pd.DataFrame()
sales_audit['row_id'] = sales_final['row_id'].values
sales_audit['receipt_number'] = df['Receipt Number'].values
sales_audit['datetime'] = sales_final['datetime'].values
sales_audit['item_name'] = df['Items'].values
sales_audit['merged_row_count'] = df['Merged_Row_Count'].values
sales_audit['reason_raw'] = df['Reason_Raw'].values
sales_audit['anomaly_flag'] = df['Anomaly_Flag'].values
sales_audit['item_flag'] = df['Item_Flag'].values
sales_audit['net_diff'] = (df['Gross Sales'] - df['Discounts'] - df['Refunds'] - df['Net Sales']).round(4).values

sales_audit_cols = [
    'row_id', 'receipt_number', 'datetime', 'item_name', 'merged_row_count',
    'reason_raw', 'anomaly_flag', 'item_flag', 'net_diff'
]
sales_audit = sales_audit[sales_audit_cols]
sales_audit.to_csv(output_sales_audit, index=False)

print(f"[9.2] File '{output_sales_audit}' berhasil disimpan: {sales_audit.shape[0]:,} baris x {sales_audit.shape[1]} kolom")


[9.2] File 'sales_audit.csv' berhasil disimpan: 61,431 baris x 9 kolom


### 9.3 Ekspor `daily_item_demand.csv` (Forecasting & BOM)
Mengekspor deret waktu harian level menu (18.423 baris: 267 hari x 69 menu) dengan pemisahan hari buka/tutup dan dekomposisi porsi.

In [30]:
# 9.3 Ekspor daily_item_demand.csv
output_demand = 'daily_item_demand.csv'

# Konstanta filter dan ambang batas
EXCLUDE_ITEM_FLAG = ['Custom Amount', 'Bundle']
EXCLUDE_ANOMALY = ['Kemungkinan Uji Coba', 'Outlier Gross']
REFUND_REASONS_REDUCE_QTY = ['Cancelled Order', 'Accidental Charge', 'Double Input', 'Item Not Ready']
MIN_TOTAL_QTY = 30

# Baris terpakai
df_used = df[~df['Item_Flag'].isin(EXCLUDE_ITEM_FLAG) & ~df['Anomaly_Flag'].isin(EXCLUDE_ANOMALY)].copy()
df_used['date'] = df_used['Datetime'].dt.strftime('%Y-%m-%d')

# Rentang tanggal kalender lengkap dari min s/d max di df
min_date = df['Datetime'].dt.date.min()
max_date = df['Datetime'].dt.date.max()
all_dates = pd.date_range(min_date, max_date, freq='D').strftime('%Y-%m-%d')
active_dates = set(df['Datetime'].dt.strftime('%Y-%m-%d').unique())
closed_dates = [d for d in all_dates if d not in active_dates]

# Semua menu unik yang memiliki baris terpakai
used_menus = sorted(df_used['Items'].unique())
menu_category = df_used.groupby('Items')['Category'].first().to_dict()

# Agregasi komponen metrik
pmt = df_used[df_used['Event Type'] == 'Payment'].groupby(['date', 'Items'])['Quantity'].sum().rename('qty_payment')
ref = df_used[(df_used['Event Type'] == 'Refund') & (df_used['Reason of Refund'].isin(REFUND_REASONS_REDUCE_QTY))].groupby(['date', 'Items'])['Quantity'].sum().rename('qty_refund')
gratis = df_used[(df_used['Event Type'] == 'Payment') & (df_used['Anomaly_Flag'] == 'Gratis/Diskon Penuh')].groupby(['date', 'Items'])['Quantity'].sum().rename('qty_gratis')
n_rcpt = df_used[df_used['Event Type'] == 'Payment'].groupby(['date', 'Items'])['Receipt Number'].nunique().rename('n_receipts')
ns = df_used.groupby(['date', 'Items'])['Net Sales'].sum().rename('net_sales')

# Pembentukan kerangka (grid) tanggal x menu
grid = pd.MultiIndex.from_product([all_dates, used_menus], names=['date', 'item_name']).to_frame().reset_index(drop=True)
grid['category'] = grid['item_name'].map(menu_category)

# Gabungkan metrik agregasi ke grid
grid = grid.merge(pmt, left_on=['date', 'item_name'], right_index=True, how='left')
grid = grid.merge(ref, left_on=['date', 'item_name'], right_index=True, how='left')
grid = grid.merge(gratis, left_on=['date', 'item_name'], right_index=True, how='left')
grid = grid.merge(n_rcpt, left_on=['date', 'item_name'], right_index=True, how='left')
grid = grid.merge(ns, left_on=['date', 'item_name'], right_index=True, how='left')

# Penentuan status hari buka vs tutup
grid['is_open'] = grid['date'].isin(active_dates)
open_mask = grid['is_open']

# Untuk hari BUKA: kombinasi tanpa penjualan diisi 0
grid.loc[open_mask, 'qty_payment'] = grid.loc[open_mask, 'qty_payment'].fillna(0)
grid.loc[open_mask, 'qty_refund'] = grid.loc[open_mask, 'qty_refund'].fillna(0)
grid.loc[open_mask, 'qty_gratis'] = grid.loc[open_mask, 'qty_gratis'].fillna(0)
grid.loc[open_mask, 'n_receipts'] = grid.loc[open_mask, 'n_receipts'].fillna(0)
grid.loc[open_mask, 'net_sales'] = grid.loc[open_mask, 'net_sales'].fillna(0)

# qty_net = qty_payment + qty_refund (pada hari buka, pada hari tutup dibiarkan NaN)
grid['qty_net'] = np.where(grid['is_open'], grid['qty_payment'] + grid['qty_refund'], np.nan)

# Fitur kalender
dt_series = pd.to_datetime(grid['date'])
grid['day_name'] = dt_series.dt.day_name()
grid['is_weekend'] = dt_series.dt.dayofweek.isin([5, 6])

# Flag menu jarang terjual (is_sparse_item)
menu_total_qty = grid.groupby('item_name')['qty_net'].sum()
sparse_menus = menu_total_qty[menu_total_qty < MIN_TOTAL_QTY].index.tolist()
grid['is_sparse_item'] = grid['item_name'].isin(sparse_menus)

# Susun kolom sesuai spesifikasi
demand_cols = [
    'date', 'item_name', 'category', 'qty_payment', 'qty_refund', 'qty_net',
    'qty_gratis', 'n_receipts', 'net_sales', 'day_name', 'is_weekend', 'is_open', 'is_sparse_item'
]
grid = grid[demand_cols]
grid.to_csv(output_demand, index=False)

print(f"[9.3] File '{output_demand}' berhasil disimpan: {grid.shape[0]:,} baris ({len(all_dates)} hari x {len(used_menus)} menu)")
print(f"      - Hari Buka  : {grid['date'][grid['is_open']].nunique()} hari")
print(f"      - Hari Tutup : {len(closed_dates)} hari ({', '.join(closed_dates)})")
print(f"      - Total qty_payment : {grid['qty_payment'].sum():,.0f} porsi")
print(f"      - Total qty_refund  : {grid['qty_refund'].sum():,.0f} porsi")
print(f"      - Total qty_net     : {grid['qty_net'].sum():,.0f} porsi")
print(f"      - Total qty_gratis  : {grid['qty_gratis'].sum():,.0f} porsi")
print(f"      - Menu Sparse (<{MIN_TOTAL_QTY} porsi): {len(sparse_menus)} menu ({', '.join(sparse_menus)})")

# Evaluasi jam transaksi pada tanggal terakhir (2026-09-24) vs rata-rata hari lain
last_date = df['Datetime'].dt.strftime('%Y-%m-%d').max()
last_day_rcpt = df[df['Datetime'].dt.strftime('%Y-%m-%d') == last_date].groupby(df['Datetime'].dt.hour)['Receipt Number'].nunique()
other_days = df[df['Datetime'].dt.strftime('%Y-%m-%d') != last_date]
other_hourly_avg = other_days.groupby([other_days['Datetime'].dt.strftime('%Y-%m-%d'), other_days['Datetime'].dt.hour])['Receipt Number'].nunique().groupby(level=1).mean()

hourly_comp = pd.DataFrame({
    f'Struk ({last_date})': last_day_rcpt,
    'Rata-rata Hari Lain': other_hourly_avg.round(1)
}).fillna(0)
print(f"\n[9.3] Pemeriksaan Jam Transaksi Tanggal Terakhir ({last_date}) vs Rata-rata Hari Lain:")
print(hourly_comp.to_string())


[9.3] File 'daily_item_demand.csv' berhasil disimpan: 18,423 baris (267 hari x 69 menu)
      - Hari Buka  : 264 hari
      - Hari Tutup : 3 hari (2026-03-22, 2026-03-23, 2026-05-27)
      - Total qty_payment : 98,236 porsi
      - Total qty_refund  : -187 porsi
      - Total qty_net     : 98,049 porsi
      - Total qty_gratis  : 4,384 porsi
      - Menu Sparse (<30 porsi): 9 menu (Air Putih, Bakso Beranak, Bingo Cone, Ceker, Choco Melt, Crispy Balls, Markisa Stik, Mie Teriyaki, Mochi Vanila)

[9.3] Pemeriksaan Jam Transaksi Tanggal Terakhir (2026-09-24) vs Rata-rata Hari Lain:
          Struk (2026-09-24)  Rata-rata Hari Lain
Datetime                                         
8                        0.0                  1.0
9                        1.0                  1.4
10                       1.0                  2.4
11                       2.0                  3.5
12                       6.0                  5.6
13                       3.0                  4.2
14             

### 9.4 Validasi Integritas Output Siap-Pakai (`assert`)
Menguji 8 asersi integritas data pada ketiga file output siap-pakai (`sales_final.csv`, `sales_audit.csv`, `daily_item_demand.csv`).

In [31]:
# 9.4 Validasi assert integritas output siap-pakai
import re

sf_read = pd.read_csv('sales_final.csv', keep_default_na=False)
sa_read = pd.read_csv('sales_audit.csv', keep_default_na=False)
did_read = pd.read_csv('daily_item_demand.csv', keep_default_na=False)

# 1. Baris sales_final = sales_audit = len(df); row_id unik dan berurutan 1..N
assert len(sf_read) == len(df), f"Baris sales_final ({len(sf_read)}) != len(df) ({len(df)})"
assert len(sa_read) == len(df), f"Baris sales_audit ({len(sa_read)}) != len(df) ({len(df)})"
assert (sf_read['row_id'].astype(int) == np.arange(1, len(df) + 1)).all(), "row_id sales_final tidak berurutan 1..N!"
assert (sa_read['row_id'].astype(int) == np.arange(1, len(df) + 1)).all(), "row_id sales_audit tidak berurutan 1..N!"
print("[ASSERT 1 PASS] Jumlah baris dan row_id sales_final & sales_audit sesuai sempurna.")

# 2. Kolom inti tanpa kosong
for col in ['datetime', 'category', 'item_name', 'quantity', 'gross_sales', 'net_sales']:
    assert (sf_read[col] == '').sum() == 0, f"Kolom inti '{col}' di sales_final memuat sel kosong!"
print("[ASSERT 2 PASS] Kolom inti sales_final 100% terisi lengkap tanpa missing values.")

# 3. Tidak ada string 'nan', 'None', 'NaN' di ketiga file
for f_name, f_df in [('sales_final.csv', sf_read), ('sales_audit.csv', sa_read), ('daily_item_demand.csv', did_read)]:
    for col in f_df.columns:
        assert (f_df[col] == 'nan').sum() == 0, f"Kolom '{col}' di {f_name} memuat teks 'nan'!"
        assert (f_df[col] == 'None').sum() == 0, f"Kolom '{col}' di {f_name} memuat teks 'None'!"
        assert (f_df[col] == 'NaN').sum() == 0, f"Kolom '{col}' di {f_name} memuat teks 'NaN'!"
print("[ASSERT 3 PASS] Tidak ada teks 'nan', 'None', atau 'NaN' di ketiga file output siap-pakai.")

# 4. max |net_diff| <= 0.5
max_net_diff = sa_read['net_diff'].astype(float).abs().max()
assert max_net_diff <= 0.5, f"max |net_diff| melebihi batas toleransi! ({max_net_diff})"
print(f"[ASSERT 4 PASS] Formula keuangan konsisten: max |net_diff| = {max_net_diff} (<= 0.5).")

# 5. Rekonsiliasi finansial sales_final vs (df_raw - dropped_nagih)
sf_gross = sf_read['gross_sales'].astype(float).sum()
sf_net = sf_read['net_sales'].astype(float).sum()
expected_raw_gross = df_raw['Gross Sales'].sum() - dropped_nagih['Gross Sales'].sum()
expected_raw_net = df_raw['Net Sales'].sum() - dropped_nagih['Net Sales'].sum()
assert abs(sf_gross - expected_raw_gross) <= 0.01, f"Gross sales sales_final selisih {abs(sf_gross - expected_raw_gross)}"
assert abs(sf_net - expected_raw_net) <= 0.01, f"Net sales sales_final selisih {abs(sf_net - expected_raw_net)}"
print("[ASSERT 5 PASS] Rekonsiliasi nominal Gross dan Net Sales sales_final cocok persis (selisih Rp 0.00).")

# 6. sum(merged_row_count) == len(df_raw) - len(dropped_nagih)
total_merged_rows = sa_read['merged_row_count'].astype(int).sum()
expected_unmerged = len(df_raw) - len(dropped_nagih)
assert total_merged_rows == expected_unmerged, f"sum(merged_row_count) ({total_merged_rows}) != expected ({expected_unmerged})"
print(f"[ASSERT 6 PASS] Total merged_row_count cocok dengan baris operasional sebelum agregasi ({total_merged_rows:,} baris).")

# 7. daily_item_demand validasi hari buka/tutup & sum(qty_net)
open_rows = did_read[did_read['is_open'].astype(bool) == True]
closed_rows = did_read[did_read['is_open'].astype(bool) == False]
for col in ['qty_payment', 'qty_refund', 'qty_net', 'qty_gratis', 'n_receipts', 'net_sales']:
    assert (open_rows[col] == '').sum() == 0, f"Kolom '{col}' pada hari buka memuat sel kosong!"
    assert (closed_rows[col] != '').sum() == 0, f"Kolom '{col}' pada hari tutup harus kosong!"
assert (open_rows['qty_net'].astype(float) < 0).sum() == 0, "Ditemukan qty_net negatif pada hari buka!"

# Bandingkan sum(qty_net) dengan hitungan mandiri dari sales_final
sf_used = sf_read[~sf_read['item_flag'].isin(EXCLUDE_ITEM_FLAG) & ~sf_read['anomaly_flag'].isin(EXCLUDE_ANOMALY)]
sf_pmt_qty = sf_used[sf_used['event_type'] == 'Payment']['quantity'].astype(float).sum()
sf_ref_qty = sf_used[(sf_used['event_type'] == 'Refund') & (sf_used['reason_of_refund'].isin(REFUND_REASONS_REDUCE_QTY))]['quantity'].astype(float).sum()
expected_total_qty_net = sf_pmt_qty + sf_ref_qty
did_total_qty_net = open_rows['qty_net'].astype(float).sum()
assert abs(did_total_qty_net - expected_total_qty_net) < 1e-4, f"sum(qty_net) mismatch: {did_total_qty_net} != {expected_total_qty_net}"
print(f"[ASSERT 7 PASS] daily_item_demand valid: sum(qty_net) = {did_total_qty_net:,.0f} porsi, hari buka/tutup konsisten.")

# 8. Bebas kolom Customer dan bebas nomor telepon pada seluruh kolom teks
phone_pattern = re.compile(r'^\+?\d{9,}$')
for f_name, f_df in [('sales_final.csv', sf_read), ('sales_audit.csv', sa_read), ('daily_item_demand.csv', did_read)]:
    assert 'Customer' not in f_df.columns, f"Kolom 'Customer' masih ada di {f_name}!"
    for col in f_df.columns:
        matches = f_df[col].astype(str).str.contains(phone_pattern).sum()
        assert matches == 0, f"Ditemukan pola nomor telepon di kolom '{col}' file {f_name} ({matches} baris)!"
print("[ASSERT 8 PASS] Terverifikasi 100% bebas dari kolom Customer dan nomor telepon di seluruh file siap-pakai.")


[ASSERT 1 PASS] Jumlah baris dan row_id sales_final & sales_audit sesuai sempurna.
[ASSERT 2 PASS] Kolom inti sales_final 100% terisi lengkap tanpa missing values.
[ASSERT 3 PASS] Tidak ada teks 'nan', 'None', atau 'NaN' di ketiga file output siap-pakai.
[ASSERT 4 PASS] Formula keuangan konsisten: max |net_diff| = 0.03 (<= 0.5).
[ASSERT 5 PASS] Rekonsiliasi nominal Gross dan Net Sales sales_final cocok persis (selisih Rp 0.00).
[ASSERT 6 PASS] Total merged_row_count cocok dengan baris operasional sebelum agregasi (63,935 baris).
[ASSERT 7 PASS] daily_item_demand valid: sum(qty_net) = 98,049 porsi, hari buka/tutup konsisten.
[ASSERT 8 PASS] Terverifikasi 100% bebas dari kolom Customer dan nomor telepon di seluruh file siap-pakai.
